# Handwritten Digit Classifier (MNIST) with PyTorch

A deep neural network (3 hidden layers + dropout) that classifies handwritten digits 0–9.

**Result:** 97.6% accuracy on the 10,000-image MNIST test set.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import random_split, DataLoader

In [2]:
# 1. Check device (GPU if available, else CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
# 2. Define transforms for MNIST: convert to Tensor, normalize
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

In [4]:
# 3. Download the MNIST dataset
full_train_dataset = torchvision.datasets.MNIST(
    root='./data', train=True, transform=transform, download=True
)
test_dataset = torchvision.datasets.MNIST(
    root='./data', train=False, transform=transform, download=True
)

100%|██████████| 9.91M/9.91M [00:00<00:00, 20.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 667kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.85MB/s]
100%|██████████| 4.54k/4.54k [00:00<?, ?B/s]


In [5]:
# 4. Split into training (50k) and validation (10k) sets
train_dataset, val_dataset = random_split(full_train_dataset, [50000, 10000])

In [6]:
# 5. Create Data Loaders (batch size = 64, as specified)
train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)

In [7]:
# 6. Define the DeepMLP class with 3 hidden layers and dropout=0.3 after first hidden layer
class DeepMLP(nn.Module):
    def __init__(self):
        super(DeepMLP, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 256)
        self.dropout = nn.Dropout(p=0.3)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, 10)

    def forward(self, x):
        x = x.view(-1, 28 * 28)  # Flatten input
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = self.fc4(x)
        return x


In [8]:
# Instantiate model, move to device (GPU or CPU)
model = DeepMLP().to(device)
print(model)

DeepMLP(
  (fc1): Linear(in_features=784, out_features=256, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=64, bias=True)
  (fc4): Linear(in_features=64, out_features=10, bias=True)
)


In [9]:
# 8. Helper function for validation/test evaluation
def evaluate(model, loader):
    model.eval()
    total = 0
    correct = 0
    total_loss = 0.0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    avg_loss = total_loss / total 
    accuracy = 100 * correct / total
    return avg_loss, accuracy


In [11]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Make sure this is defined too
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Define optimizer


In [12]:
#Main training loop for 5 epochs (as specified)
num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    # Validation phase
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)  # Multiply by batch size
            
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_accuracy = 100 * correct / total
    avg_train_loss = running_loss / len(train_loader.dataset)
    avg_val_loss = val_loss / total 

    print(f"Epoch [{epoch+1}/{num_epochs}], "
          f"Train Loss: {avg_train_loss:.4f}, "
          f"Val Loss: {avg_val_loss:.4f}, "
          f"Val Accuracy: {val_accuracy:.2f}%")

Epoch [1/5], Train Loss: 0.3225, Val Loss: 0.1608, Val Accuracy: 95.04%
Epoch [2/5], Train Loss: 0.1512, Val Loss: 0.1162, Val Accuracy: 96.76%
Epoch [3/5], Train Loss: 0.1169, Val Loss: 0.0963, Val Accuracy: 97.21%
Epoch [4/5], Train Loss: 0.1011, Val Loss: 0.0801, Val Accuracy: 97.60%
Epoch [5/5], Train Loss: 0.0872, Val Loss: 0.0859, Val Accuracy: 97.38%


In [15]:
# 10. Final test evaluation
test_loss, test_accuracy = evaluate(model, test_loader)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.2f}%")


Test Loss: 0.0826, Test Accuracy: 97.60%
